# 第 3 天：转换成 YOLO 格式，并且不让测试集泄漏

这一步决定了你之后所有分数可不可信。先亲眼看看随机划分会出什么问题。

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sonar.plotting import setup_cjk_font
plt.rcParams["figure.dpi"] = 110
setup_cjk_font()   # 服务器上没有中文字体时会提示安装方法

# 下载了真实数据就改成 data/raw/UATD_Training；没下载就先用合成数据（python scripts/make_synthetic.py）
DATA_ROOT = REPO / "data" / "raw" / "UATD_Training"
if not DATA_ROOT.exists():
    DATA_ROOT = REPO / "data" / "synthetic" / "UATD_Training"
print("使用数据：", DATA_ROOT)

In [ ]:
from sonar.uatd import find_pairs, collect_classes
from sonar.split import random_split, block_split, leakage_report
anns = find_pairs(DATA_ROOT)
classes = collect_classes(anns)
rows = []
for name, (tr, va) in {
    "随机划分": random_split(anns, 0.2),
    "连续块划分 block=25": block_split(anns, 0.2, block_size=25),
    "连续块划分 block=50": block_split(anns, 0.2, block_size=50),
}.items():
    r = leakage_report(tr, va)
    rows.append({"划分": name, "训练": len(tr), "验证": len(va),
                 "平均最高相似度": round(r["mean_best_similarity"], 3),
                 "近似重复比例": f'{r["near_duplicate_ratio"]:.1%}'})
pd.DataFrame(rows)

> 问题：随机划分的「平均最高相似度」比连续块划分高多少？用随机划分算出来的验证分数会偏高还是偏低？

## 转换（基线：只做线性缩放，相当于当成光学图）

In [ ]:
from functools import partial
from sonar.convert import write_split, write_data_yaml
from sonar.preprocess import sonar_preprocess
OUT = REPO / "data" / "yolo_raw"
train, val = block_split(anns, 0.2, block_size=50)
tf = partial(sonar_preprocess, mode="raw")
write_split(train, "train", OUT, classes, tf)
write_split(val, "val", OUT, classes, tf)
print(write_data_yaml(OUT, classes, has_test=False).read_text())
# 有测试集时直接用命令行：python scripts/convert_to_yolo.py ... --test-roots ...（见教程）

## 反向检查：把 YOLO 标签画回图上，确认坐标没转错

In [ ]:
import cv2
img_p = sorted((OUT / "images" / "val").glob("*.png"))[0]
lbl_p = OUT / "labels" / "val" / f"{img_p.stem}.txt"
im = cv2.cvtColor(cv2.imread(str(img_p), cv2.IMREAD_GRAYSCALE), cv2.COLOR_GRAY2RGB)
H, W = im.shape[:2]
for line in lbl_p.read_text().splitlines():
    k, cx, cy, w, h = map(float, line.split())
    x1, y1, x2, y2 = int((cx - w / 2) * W), int((cy - h / 2) * H), int((cx + w / 2) * W), int((cy + h / 2) * H)
    cv2.rectangle(im, (x1, y1), (x2, y2), (255, 80, 80), 1)
    cv2.putText(im, classes[int(k)], (x1, max(10, y1 - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 80, 80), 1)
plt.figure(figsize=(6, 6)); plt.imshow(im); plt.axis("off"); plt.title(img_p.name); plt.show()

> 问题：框和目标对得上吗？对不上的话，最常见的原因是宽高写反或忘了归一化。